## Data Loading

This notebook can be used to load the raw data from the [RTR Linguistic API](https://developer.srgssr.ch/en/apis/rtr-linguistic). Note that you need to create an account on the site first and retrieve your credentials. Then you need to type in your credentials below:

In [ ]:
import shutil
import tarfile
import traceback
from constants import RAW_DATA_ROOT
from RtrDataLoader import RTRLinguisticDownloader

CONSUMER_KEY = ""
CONSUMER_SECRET = ""

if not CONSUMER_KEY or not CONSUMER_SECRET:
  raise Exception("Please enter your credentials")

After you entered your credentials you can run the data loader to download the data.

In [ ]:
print("RTR Linguistic Data Downloader")
print("=" * 60)

try:    
  downloader = RTRLinguisticDownloader(CONSUMER_KEY, CONSUMER_SECRET)
  folder = RAW_DATA_ROOT
  downloader.get_all_data(folder)
except Exception as e:
  print(f"\nError: {e}")
  traceback.print_exc()

The files will be downloaded into a folder called `download_*` since this folder is not needed we will move the data directly into the `raw-data` directory and then remove the intermediate `download_*` folder.


In [ ]:
if not RAW_DATA_ROOT.exists():
    raise FileNotFoundError(f"Directory {RAW_DATA_ROOT} does not exist.")

# Find all directories matching download_*
download_folders = sorted(RAW_DATA_ROOT.glob("download_*"))

if not download_folders:
    print("No download_* folders found in raw-data.")
else:
    for folder in download_folders:
        if not folder.is_dir():
            continue

        # Get list of items inside the download folder
        items = list(folder.iterdir())
        if not items:
            print(f"{folder.name} is empty, removing it.")
            folder.rmdir()
            continue

        print(f"Moving contents of {folder.name} to {RAW_DATA_ROOT}...")
        for item in items:
            dest = RAW_DATA_ROOT / item.name
            # If a file with the same name already exists, append a suffix
            if dest.exists():
                counter = 1
                stem = item.stem
                suffix = item.suffix
                while dest.exists():
                    dest = RAW_DATA_ROOT / f"{stem}_{counter}{suffix}"
                    counter += 1
                print(f"  Renaming {item.name} to {dest.name} (conflict)")
            shutil.move(str(item), str(dest))

        # Remove the now-empty folder
        folder.rmdir()
        print(f"Removed {folder.name}")

print("Done.")

Now the idiom data is still compressed into `.tgz` files which we need to unpack and then we can delete the `.tgz` files. After doing this the `data` folder will look the following:

```
raw-data/
├── rm-cc-2021-05-28/             # Rumantsch Grischun
│   ├── train.tsv
│   ├── validated.tsv
│   ├── test.tsv
│   └── clips/
│       └── *.wav
├── rmsursilv-cc-2021-05-28/      # Sursilvan
│   └── ... (same structure)
├── rmvallader-cc-2021-05-28/     # Vallader
├── rmputer-cc-2021-06-11/        # Puter
├── rmsutsilv-cc-2022-05-18/      # Sutsilvan
└── rmsursiv-cc-2021-12-23/       # Surmiran
.gitkeep
```

In [ ]:
if not RAW_DATA_ROOT.exists():
    raise FileNotFoundError("raw-data folder not found.")

tgz_files = list(RAW_DATA_ROOT.glob("*.tgz"))
if not tgz_files:
    print("No .tgz files found. They may have been extracted already.")
else:
    print(f"Found {len(tgz_files)} archives to extract.")
    for tgz in tgz_files:
        print(f"  Extracting {tgz.name} ...")
        with tarfile.open(tgz, "r:gz") as tar:
            tar.extractall(path=RAW_DATA_ROOT)
    print("Extraction complete.")

    for tgz in tgz_files:
        tgz.unlink()
        print(f"  Removed {tgz.name}")